# HDB monthly rent: final model, end to end

Runs the same stages as `python main.py`, one cell each, and writes `submission.csv`. All logic is in `modules/`; this notebook only calls it and shows the intermediate results.

**Model.** Predict the residual `rent − anchor`, where the anchor is the official quarterly median rent of the flat's town and flat type, with LightGBM, XGBoost and CatBoost, then add the anchor back and blend the three. Block-level target encodings are shrunk hierarchically and computed out of fold. Public leaderboard: **433.65480**.

**Honest anchor.** The published medians for quarters after the last labelled one (2025-Q1) aggregate test-period rents, which would not exist at forecast time. For test rows after 2025-Q1 the benchmark is frozen at the 2025-Q1 value and grown by the recent national rate. This costs about 0.05 RMSE on the public leaderboard (433.607 with the published values) and removes the leak.

| Stage | Module |
|---|---|
| 1. Cleaning and features | `cleaning.py`, `features.py` |
| 2. Honest anchor | `anchor.py` |
| 3. Blend weights (out-of-time fit) | `models.py` |
| 4. 5-fold residual models | `models.py` |
| 5. Submission file | `pipeline.py` |

Runtime is about 15–25 minutes, nearly all in stages 3–4. The earlier single-LightGBM pipeline (validation RMSE 451.87) is kept in `baseline_main.ipynb`.

In [1]:
import os, sys
sys.path.insert(0, 'modules')
import numpy as np, pandas as pd
import config
import anchor
import models
import pipeline as pipe

print('project root:', config.PROJECT_ROOT)
print('seed:', config.RANDOM_SEED, '| folds:', config.N_FOLDS, '| last labelled quarter:', config.CUTOFF_QUARTER)

project root: /Users/aayushmajumdar/Desktop/CS5228-knowledge-discovery-and-data-mining-rel/nus-cs5228/project/KDDM-project
seed: 42 | folds: 5 | last labelled quarter: 2025-Q1


## 1. Cleaning and features

Raw `train.csv` / `test.csv` plus the auxiliary files in `data/auxiliary/` become one feature table each:

- spellings of town, flat type and model unified; `-1` sentinels in the block file imputed from the street, then the town median;
- exact duplicate train rows kept (concurrent tenancies of identical flats);
- structural features, distances and counts to MRT stations (all, operating, planned, interchange), malls and schools (Haversine `BallTree`), COE and bank-stock series;
- the quarterly town × flat-type median rent, with fallbacks for empty cells, as `median_rent_benchmark`;
- five hierarchical target encodings of the residual (town, block, block × flat type, street, subzone), shrunk with empirical-Bayes constants 30/10/5/15/20. Train rows get 5-fold out-of-fold values; test rows use maps fitted on all of train.

In [2]:
train, test_published = pipe.build_features()
print('train', train.shape, '| test', test_published.shape)
train[['RENT_APPROVAL_DATE', 'town_clean', 'flat_type_clean', 'MONTHLY_RENT', 'median_rent_benchmark', 'te_block', 'dist_nearest_mrt']].head()

   [Spatial FE v2] Building BallTree spatial features for unique blocks...


   [Spatial FE v2] Completed 20 spatial features for 9456 blocks.
   [Macro FE v2] Building quarterly HDB median rent benchmarks with trend extrapolation...


   [Target Encoding v2] Computing 5-Fold OOF Hierarchical Bayesian Target Encodings...



[Feature Integrity Audit v2]:
Train Shape: (150000, 70) | Nulls: 0
Test Shape:  (50000, 69) | Nulls: 0
train (150000, 70) | test (50000, 69)


,RENT_APPROVAL_DATE,town_clean,flat_type_clean,MONTHLY_RENT,median_rent_benchmark,te_block,dist_nearest_mrt
0,2021-01,punggol,4-room,1800,2000.0,-36.520825,0.556642
1,2021-01,sengkang,5-room,2000,2000.0,1.484268,0.634370
2,2021-01,queenstown,3-room,2000,1900.0,38.762931,0.625890
3,2021-01,sengkang,4-room,1400,2000.0,5.766619,2.069777
4,2021-01,sembawang,4-room,1800,1850.0,-22.181635,0.762312


## 2. Honest test-quarter anchor

Train quarters all end at 2025-Q1, so train is unchanged. Test rows from 2025-Q2 on take the 2025-Q1 benchmark of their town and flat type times `exp(g·k)`, with `k` quarters after the cutoff and `g` the mean quarterly log growth of the national index over the last four labelled quarters.

In [3]:
test = pipe.apply_honest_anchor(test_published)
print(f'drift g = {anchor.drift(config.CUTOFF_QUARTER):+.4f} per quarter')
cmp = pd.DataFrame({'published': test_published.groupby('approval_quarter')['median_rent_benchmark'].mean(),
                    'honest': test.groupby('approval_quarter')['median_rent_benchmark'].mean(),
                    'rows': test.groupby('approval_quarter').size()}).round(1)
cmp['difference'] = (cmp['honest'] - cmp['published']).round(1)
cmp

   [Macro FE v2] Building quarterly HDB median rent benchmarks with trend extrapolation...
drift g = +0.0055 per quarter


,published,honest,rows,difference
approval_quarter,,,,
2025-Q1,3227.6,3227.6,380,0.0
2025-Q2,3238.6,3241.3,9720,2.7
2025-Q3,3250.4,3254.6,9666,4.2
2025-Q4,3222.1,3265.4,8999,43.3
2026-Q1,3234.5,3277.6,9091,43.1
2026-Q2,3280.9,3310.6,9110,29.7
2026-Q3,3303.3,3332.0,3034,28.7


## 3. Blend weights

One out-of-time fit (train through 2024-06, validate 2024-07 onward) of the three models; the weights minimise the blend RMSE on that validation window, constrained to be non-negative and sum to 1. The submitted run gave about 0.545 / 0.075 / 0.380.

In [4]:
weights, oot_rmse = models.fit_blend_weights(train)
print('weights (lgb, xgb, cb):', np.round(weights, 4))
print('out-of-time RMSE:', {k: round(v, 2) for k, v in oot_rmse.items()})

/Users/aayushmajumdar/Desktop/CS5228-knowledge-discovery-and-data-mining-rel/nus-cs5228/myenv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008348 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7114
[LightGBM] [Info] Number of data points in the train set: 124016, number of used features: 63
[LightGBM] [Info] Start training from score -13.077653


weights (lgb, xgb, cb): [0.545  0.0753 0.3797]
out-of-time RMSE: {'lgb': 450.31, 'xgb': 451.09, 'cb': 451.32, 'blend': 449.26}


## 4. Five-fold residual models

Shuffled 5-fold over all 150,000 train rows. Each fold fits the three models (early stopping on that fold's validation split); test predictions are the average over folds, then blended with the weights above. The out-of-fold RMSE is a rough guide only: it is a random split, not a forecast over time, and early stopping reuses the validation fold, so it is optimistic. The submitted run reported a blended OOF RMSE of about 471.

In [5]:
out = models.train_five_fold(train, test, weights)
print('OOF RMSE:', {k: round(v, 2) for k, v in out['oof_rmse'].items()})

/Users/aayushmajumdar/Desktop/CS5228-knowledge-discovery-and-data-mining-rel/nus-cs5228/myenv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005700 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7203
[LightGBM] [Info] Number of data points in the train set: 120000, number of used features: 63
[LightGBM] [Info] Start training from score -22.642198


fold 1/5: lgb 471.33  xgb 471.84  cb 471.09


/Users/aayushmajumdar/Desktop/CS5228-knowledge-discovery-and-data-mining-rel/nus-cs5228/myenv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006703 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7197
[LightGBM] [Info] Number of data points in the train set: 120000, number of used features: 63
[LightGBM] [Info] Start training from score -23.042585


...

fold 4/5: lgb 471.27  xgb 471.69  cb 470.71


/Users/aayushmajumdar/Desktop/CS5228-knowledge-discovery-and-data-mining-rel/nus-cs5228/myenv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008631 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7199
[LightGBM] [Info] Number of data points in the train set: 120000, number of used features: 63
[LightGBM] [Info] Start training from score -22.416804


fold 5/5: lgb 471.98  xgb 472.26  cb 471.89


OOF RMSE: {'lgb': 471.91, 'xgb': 472.11, 'cb': 471.51, 'blend': 470.99}


## 5. Submission

`Id` is the row number of `test.csv`; `Predicted` is the weighted blend.

In [6]:
submission = pipe.write_submission(out['blend'], 'submission.csv')
print(submission.shape)
submission.describe().round(1)

(50000, 2)


,Id,Predicted
count,50000.0,50000.0
mean,24999.5,3192.5
std,14433.9,477.3
min,0.0,1576.0
25%,12499.8,2847.5
50%,24999.5,3179.8
75%,37499.2,3466.4
max,49999.0,5556.3


## Reproducibility check

Compares against the file that scored 433.65480 if it is available (`expt-5/outputs/submission_v2_honest.csv`); skipped otherwise. Seeds are fixed and the models use 4 threads as in the submitted run, so the two should agree closely but need not match to the last digit on another machine or library version.

In [7]:
ref_path = 'expt-5/outputs/submission_v2_honest.csv'
if os.path.exists(ref_path):
    ref = pd.read_csv(ref_path)
    d = submission['Predicted'].values - ref['Predicted'].values
    print(f'corr {np.corrcoef(submission["Predicted"], ref["Predicted"])[0, 1]:.6f} | mean abs diff {np.abs(d).mean():.3f} | max abs diff {np.abs(d).max():.3f}')
else:
    print('reference file not present, skipped')

corr 1.000000 | mean abs diff 0.000 | max abs diff 0.000
